# Stage 3 — Prediction models

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Clone or update the project repository
import os
from google.colab import userdata
token = userdata.get('GH_TOKEN')
REPO = '/content/tevzi'
if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/emirsalihoglu/tevzi.git {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log --oneline -1

364f5ba (HEAD -> main, origin/main, origin/HEAD) feat: add GBM training with inner-holdout early stopping


In [10]:
# Make the package importable and prepare Drive folders
import sys
from pathlib import Path
sys.path.insert(0, f'{REPO}/src')
DRIVE = Path('/content/drive/MyDrive/Tevzi')
for sub in ['data/raw', 'data/interim', 'data/processed',
            'results/models', 'results/figures', 'results/tables']:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
import tevzi
print('tevzi package:', tevzi.__file__)
print('Drive folder:', DRIVE)

tevzi package: /content/tevzi/src/tevzi/__init__.py
Drive folder: /content/drive/MyDrive/Tevzi


In [11]:
# Load processed data and verify shapes and keys
import json, gc
import numpy as np
import pandas as pd
import psutil

PROC = DRIVE / 'data/processed'
jobs    = pd.read_parquet(PROC / 'jobs_flags.parquet')
targets = pd.read_parquet(PROC / 'targets.parquet')
m1      = pd.read_parquet(PROC / 'm1_features.parquet')
grid    = pd.read_parquet(PROC / 'm2_grid.parquet')
snap    = pd.read_parquet(PROC / 'm2_snapshots.parquet')
with open(PROC / 'm2_capacities.json') as f:
    caps = json.load(f)

expected = {'jobs': (jobs, 1_055_501), 'targets': (targets, 714_273),
            'm1': (m1, 714_273), 'grid': (grid, 19_858), 'snap': (snap, 714_273)}
for name, (df, n_rows) in expected.items():
    assert len(df) == n_rows, f'{name}: {len(df):,} rows, expected {n_rows:,}'
    print(f'{name:8s} {df.shape}')
assert grid.shape[1] == 28 and snap.shape[1] == 15 and m1.shape[1] == 47
assert set(m1['job_name']) == set(targets['job_name']) == set(snap['job_name'])
print('All shape and key checks passed')
print(f'RAM used: {psutil.Process().memory_info().rss / 1e9:.1f} GB')

for name, df in [('targets', targets), ('m1', m1), ('snap', snap)]:
    print(f'\n{name} columns:', list(df.columns))

jobs     (1055501, 15)
targets  (714273, 22)
m1       (714273, 47)
grid     (19858, 28)
snap     (714273, 15)
All shape and key checks passed
RAM used: 2.8 GB

targets columns: ['job_name', 'main_task', 'main_inst_num', 'main_plan_cpu', 'main_plan_mem', 'main_plan_gpu', 'main_gpu_type', 'gpu_util_max', 'max_mem_max', 'n_sensor', 'n_attempts', 'run_time', 'log_run_time', 'wait_time', 'gpu_ratio', 'mem_ratio', 'has_usage', 'usage_coverage', 'submit_day', 'period', 'purged', 'in_burst']

m1 columns: ['job_name', 'main_plan_gpu', 'main_plan_cpu', 'main_plan_mem', 'main_inst_num', 'req_gpu_total', 'req_cpu_total', 'req_mem_total', 'main_gpu_type', 'gpu_fractional', 'mem_per_gpu', 'cpu_per_gpu', 'n_tasks', 'has_ps', 'has_evaluator', 'n_ps_inst', 'total_inst', 'main_task', 'user', 'workload', 'workload_missing', 'grp_rt_n', 'grp_rt_mean', 'grp_rt_std', 'grp_rt_last', 'grp_rt_last2', 'usr_rt_n', 'usr_rt_mean', 'usr_rt_std', 'usr_rt_last', 'usr_rt_last2', 'grp_use_n', 'grp_gpu_mean', 'grp_mem_m

In [ ]:
# Disk usage of project folders (to plan cleanup)
!du -sh "{DRIVE}/data/raw" "{DRIVE}/data/interim" "{DRIVE}/data/processed" "{DRIVE}/results"
!du -sh "{DRIVE}"/data/raw/* "{DRIVE}"/data/interim/* 2>/dev/null | sort -rh | head -20

1.4G	/content/drive/MyDrive/Tevzi/data/raw
1.2G	/content/drive/MyDrive/Tevzi/data/interim
204M	/content/drive/MyDrive/Tevzi/data/processed
18K	/content/drive/MyDrive/Tevzi/results
663M	/content/drive/MyDrive/Tevzi/data/raw/pai_instance_table.tar.gz
520M	/content/drive/MyDrive/Tevzi/data/interim/pai_instance_table.parquet
388M	/content/drive/MyDrive/Tevzi/data/raw/pai_sensor_table.tar.gz
321M	/content/drive/MyDrive/Tevzi/data/interim/pai_sensor_table.parquet
198M	/content/drive/MyDrive/Tevzi/data/raw/pai_machine_metric.tar.gz
175M	/content/drive/MyDrive/Tevzi/data/interim/pai_machine_metric.parquet
60M	/content/drive/MyDrive/Tevzi/data/raw/pai_job_table.tar.gz
53M	/content/drive/MyDrive/Tevzi/data/raw/pai_group_tag_table.tar.gz
51M	/content/drive/MyDrive/Tevzi/data/interim/pai_job_table.parquet
45M	/content/drive/MyDrive/Tevzi/data/interim/pai_group_tag_table.parquet
34M	/content/drive/MyDrive/Tevzi/data/raw/pai_task_table.tar.gz
27M	/content/drive/MyDrive/Tevzi/data/interim/pai_task_ta

In [ ]:
# M3 probe, part 1: as-of group history (jobs finished strictly before submission)
INTERIM = DRIVE / 'data/interim'
job_tab = pd.read_parquet(INTERIM / 'pai_job_table.parquet', columns=['job_name', 'inst_id'])
gtag = (pd.read_parquet(INTERIM / 'pai_group_tag_table.parquet', columns=['inst_id', 'group'])
        .drop_duplicates('inst_id'))

h = (targets[['job_name', 'log_run_time', 'gpu_ratio', 'period', 'purged']]
     .merge(jobs[['job_name', 'submit_time', 'end_time']], on='job_name', how='left')
     .merge(job_tab, on='job_name', how='left')
     .merge(gtag, on='inst_id', how='left'))
assert len(h) == 714_273
print(f"group missing: {h['group'].isna().mean():.4%}")
h = h.dropna(subset=['group']).reset_index(drop=True)
assert np.isfinite(h[['log_run_time', 'submit_time', 'end_time']].to_numpy()).all()

# Sort finished jobs by (group, end); history = ends strictly before submission.
# Flooring both times is conservative: same-second ends are excluded.
SCALE = 10**8
assert h['end_time'].max() < SCALE
g = h['group'].astype('category').cat.codes.to_numpy().astype(np.int64)
end_key = g * SCALE + np.floor(h['end_time'].to_numpy()).astype(np.int64)
sub_key = g * SCALE + np.floor(h['submit_time'].to_numpy()).astype(np.int64)
order = np.argsort(end_key, kind='stable')
sorted_key = end_key[order]
lo = np.searchsorted(sorted_key, g * SCALE, side='left')
hi = np.searchsorted(sorted_key, sub_key, side='left')
h['n_hist'] = hi - lo

bins = [-1, 0, 1, 4, 9, 49, np.inf]
labels = ['0', '1', '2-4', '5-9', '10-49', '50+']
dist = pd.crosstab(pd.cut(h['n_hist'], bins=bins, labels=labels), h['period'], normalize='columns')
print('\nShare of jobs by finished group jobs before submission (%):')
print((dist * 100).round(1))
print(f'RAM used: {psutil.Process().memory_info().rss / 1e9:.1f} GB')

group missing: 0.1025%

Share of jobs by finished group jobs before submission (%):
period  test  train   val  warmup
n_hist                           
0       28.4   29.5  29.4    36.1
1        4.7    2.5   5.6     6.8
2-4      7.2    3.9   9.5    15.8
5-9      5.6    4.5   6.2    20.7
10-49   30.4   47.9  39.8    10.9
50+     23.6   11.7   9.5     9.8
RAM used: 2.1 GB


**M3 probe — decision criterion (written before results).**
Build the M3 sequence modality if adding the last-10 group history sequence
(flattened, to GBM) reduces validation MAE of log run time by **≥ 2% relative**
on jobs with **≥ 5** finished group jobs, averaged over 3 seeds, and the gain
is larger than 2× the seed standard deviation. Otherwise M3 is not added to
the core plan and the result is reported.

In [ ]:
# M3 probe, part 2: does the ordered sequence add information beyond aggregates? (train -> val)
import warnings
import lightgbm as lgb

K = 10
lrt_s = h['log_run_time'].to_numpy()[order]
gpu_s = h['gpu_ratio'].to_numpy()[order]
end_s = h['end_time'].to_numpy()[order]
sub = h['submit_time'].to_numpy()

idx = hi[:, None] - np.arange(1, K + 1)[None, :]      # column 0 = most recently finished
valid = idx >= lo[:, None]
idx = np.where(valid, idx, 0)
assert ((sub[:, None] - end_s[idx])[valid] > 0).all()   # leakage guard
seq = {'lrt': np.where(valid, lrt_s[idx], np.nan),
       'gpu': np.where(valid, gpu_s[idx], np.nan),
       'gap': np.where(valid, np.log1p(np.maximum(sub[:, None] - end_s[idx], 0)), np.nan)}

# Aggregates over the full history (same information M1 summarises)
cs = np.concatenate([[0.0], np.cumsum(lrt_s)])
cs2 = np.concatenate([[0.0], np.cumsum(lrt_s ** 2)])
n = (hi - lo).astype(float)
with np.errstate(invalid='ignore', divide='ignore'), warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    mean_all = (cs[hi] - cs[lo]) / n
    std_all = np.sqrt(np.maximum((cs2[hi] - cs2[lo]) / n - mean_all ** 2, 0))
    last2 = np.nanmean(seq['lrt'][:, :2], axis=1)
agg = pd.DataFrame({'h_n': n, 'h_mean': mean_all, 'h_std': std_all, 'h_last': seq['lrt'][:, 0],
                    'h_last2': last2, 'h_gap_last': seq['gap'][:, 0]})
seq_df = pd.DataFrame(np.hstack([seq[k] for k in seq]),
                      columns=[f's_{k}_{i}' for k in seq for i in range(K)])

X_m1 = h[['job_name']].merge(m1, on='job_name', how='left').drop(columns='job_name')
assert len(X_m1) == len(h)
for c in X_m1.columns:
    if X_m1[c].dtype == object:
        X_m1[c] = X_m1[c].astype('category')
XA = pd.concat([X_m1, agg], axis=1)
XB = pd.concat([XA, seq_df], axis=1)

tr = ((h['period'] == 'train') & ~h['purged']).to_numpy()
va = ((h['period'] == 'val') & ~h['purged']).to_numpy()
assert tr.sum() > 280_000 and va.sum() > 88_000, (tr.sum(), va.sum())
y = h['log_run_time'].to_numpy()
nh = h['n_hist'].to_numpy()[va]
subsets = {'all': nh >= 0, 'n=0': nh == 0, '1-4': (nh >= 1) & (nh <= 4),
           '>=5': nh >= 5, '>=10': nh >= 10}

params = dict(objective='l1', learning_rate=0.05, num_leaves=127, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              n_estimators=800, verbose=-1)
rows, models = [], {}
for name, X in [('A_aggregates', XA), ('B_plus_sequence', XB)]:
    for seed in (0, 1, 2):
        model = lgb.LGBMRegressor(**params, random_state=seed).fit(X[tr], y[tr])
        p = model.predict(X[va])
        for s, m in subsets.items():
            rows.append((name, seed, s, np.abs(p[m] - y[va][m]).mean(), int(m.sum())))
        models[name] = model
        print(f'{name} seed {seed} done')

res = pd.DataFrame(rows, columns=['model', 'seed', 'subset', 'mae', 'n'])
res.to_csv(DRIVE / 'results/tables/m3_probe.csv', index=False)
mean = res.groupby(['subset', 'model'])['mae'].mean().unstack()
std = res.groupby(['subset', 'model'])['mae'].std().unstack()
mean['gain_%'] = (1 - mean['B_plus_sequence'] / mean['A_aggregates']) * 100
mean['2x_seed_std_%'] = 2 * std.max(axis=1) / mean['A_aggregates'] * 100
mean['n'] = res.groupby('subset')['n'].first()
print('\nValidation MAE (log run time), mean of 3 seeds:')
print(mean.loc[list(subsets)].round(4))

imp = pd.Series(models['B_plus_sequence'].booster_.feature_importance('gain'), index=XB.columns)
print(f"\nSequence share of total gain (model B): {imp[seq_df.columns].sum() / imp.sum():.1%}")
print(imp.sort_values(ascending=False).head(15).round(0))

A_aggregates seed 0 done
A_aggregates seed 1 done
A_aggregates seed 2 done
B_plus_sequence seed 0 done
B_plus_sequence seed 1 done
B_plus_sequence seed 2 done

Validation MAE (log run time), mean of 3 seeds:
model   A_aggregates  B_plus_sequence  gain_%  2x_seed_std_%      n
subset                                                             
all           0.6491           0.6332  2.4475         2.3082  88589
n=0           0.9650           0.9543  1.1055         4.1513  25983
1-4           1.1112           1.0667  4.0095         2.6953  13378
>=5           0.3568           0.3459  3.0409         5.5154  49228
>=10          0.3149           0.3074  2.3881         3.2127  43703

Sequence share of total gain (model B): 6.1%
grp_rt_last2    1761238.0
grp_rt_mean     1148469.0
user             870671.0
h_last2          598403.0
h_mean           483229.0
grp_rt_last      371745.0
usr_rt_mean      331991.0
usr_rt_last2     272046.0
usr_rt_last      129932.0
grp_rt_std        90880.0
hour_sin  

**M3 probe (usage) — decision criterion (written before results).**
Same design as the run-time probe, target = GPU usage ratio (jobs with usage data).
Build M3 for the usage head if model B reduces validation MAE of gpu_ratio by
**≥ 2% relative** on jobs with **≥ 5** finished group jobs (mean of 3 seeds) and the
gain is larger than 2× the seed standard deviation (same definition as before).
The run-time probe result stands regardless and is reported.

In [ ]:
# M3 probe, part 3: GPU usage ratio target (criterion fixed in the cell above)
y_g = h['gpu_ratio'].to_numpy()
ok = np.isfinite(y_g)
tr_g, va_g = tr & ok, va & ok
print(f'train {tr_g.sum():,}  val {va_g.sum():,}')
nh_g = h['n_hist'].to_numpy()[va_g]
subsets_g = {'all': nh_g >= 0, 'n=0': nh_g == 0, '1-4': (nh_g >= 1) & (nh_g <= 4),
             '>=5': nh_g >= 5, '>=10': nh_g >= 10}

rows_g = []
for name, X in [('A_aggregates', XA), ('B_plus_sequence', XB)]:
    for seed in (0, 1, 2):
        model = lgb.LGBMRegressor(**params, random_state=seed).fit(X[tr_g], y_g[tr_g])
        p = model.predict(X[va_g])
        for s, m in subsets_g.items():
            rows_g.append((name, seed, s, np.abs(p[m] - y_g[va_g][m]).mean(), int(m.sum())))
        print(f'{name} seed {seed} done')

res_g = pd.DataFrame(rows_g, columns=['model', 'seed', 'subset', 'mae', 'n'])
res_g.to_csv(DRIVE / 'results/tables/m3_probe_gpu.csv', index=False)
mean_g = res_g.groupby(['subset', 'model'])['mae'].mean().unstack()
std_g = res_g.groupby(['subset', 'model'])['mae'].std().unstack()
mean_g['gain_%'] = (1 - mean_g['B_plus_sequence'] / mean_g['A_aggregates']) * 100
mean_g['2x_seed_std_%'] = 2 * std_g.max(axis=1) / mean_g['A_aggregates'] * 100
mean_g['n'] = res_g.groupby('subset')['n'].first()
print('\nValidation MAE (gpu_ratio), mean of 3 seeds:')
print(mean_g.loc[list(subsets_g)].round(4))

train 270,750  val 87,470
A_aggregates seed 0 done
A_aggregates seed 1 done
A_aggregates seed 2 done
B_plus_sequence seed 0 done
B_plus_sequence seed 1 done
B_plus_sequence seed 2 done

Validation MAE (gpu_ratio), mean of 3 seeds:
model   A_aggregates  B_plus_sequence  gain_%  2x_seed_std_%      n
subset                                                             
all           0.0783           0.0780  0.3640         1.4787  87470
n=0           0.1139           0.1135  0.3161         1.5187  25635
1-4           0.0947           0.0956 -1.0273         4.2319  13229
>=5           0.0551           0.0545  1.0665         1.2504  48606
>=10          0.0584           0.0578  1.1694         0.7875  43132


In [ ]:
# Run all unit tests, then inspect history feature names and scale before baselines
!cd {REPO} && PYTHONPATH=src python -m pytest -q tests/

hist_cols = [c for c in m1.columns if c.startswith(('grp_', 'usr_'))]
print('\nHistory columns:', hist_cols)
print('\nScale check (should be comparable to log_run_time):')
cmp = pd.concat([targets['log_run_time'].describe(),
                 m1['grp_rt_last2'].describe(), m1['usr_rt_last2'].describe()], axis=1)
print(cmp.round(2))

.............................                                            [100%]
29 passed in 4.97s

History columns: ['grp_rt_n', 'grp_rt_mean', 'grp_rt_std', 'grp_rt_last', 'grp_rt_last2', 'usr_rt_n', 'usr_rt_mean', 'usr_rt_std', 'usr_rt_last', 'usr_rt_last2', 'grp_use_n', 'grp_gpu_mean', 'grp_mem_mean', 'grp_gpu_zero_share', 'usr_use_n', 'usr_gpu_mean', 'usr_mem_mean', 'usr_gpu_zero_share', 'grp_n_prev_submit', 'grp_seen_before', 'grp_time_since_prev', 'grp_submit_last_1h']

Scale check (should be comparable to log_run_time):
       log_run_time  grp_rt_last2  usr_rt_last2
count     714273.00     497308.00     711994.00
mean           6.41          6.50          6.50
std            2.20          2.09          2.05
min            0.69          0.69          0.69
25%            4.64          4.98          4.91
50%            6.28          6.33          6.42
75%            8.17          8.24          8.08
max           13.35         13.18         13.32


In [ ]:
# Stage 3.1: baselines under the pre-registered metrics (validation period only)
from tevzi.evaluation.metrics import SubsetThresholds, make_subsets, runtime_report, usage_report

need = {'purged', 'in_burst', 'run_time', 'usage_coverage'}
assert need <= set(targets.columns), need - set(targets.columns)
cols_m1 = ['job_name', 'grp_rt_n', 'grp_rt_last2', 'usr_rt_last2',
           'grp_gpu_mean', 'usr_gpu_mean', 'grp_mem_mean', 'usr_mem_mean']
ev = (targets.merge(m1[cols_m1], on='job_name', how='left')
             .merge(snap[['job_name', 'submit', 'pool_key', 'pool_pressure']], on='job_name', how='left'))
assert len(ev) == len(targets)
tr = ((ev['period'] == 'train') & ~ev['purged']).to_numpy()
va = ((ev['period'] == 'val') & ~ev['purged']).to_numpy()

thr = SubsetThresholds.fit(ev.loc[tr, 'pool_pressure'])
V = ev[va].reset_index(drop=True)
subs = make_subsets(V, thr)
print(f"busy threshold (train q75 of pool_pressure): {thr.busy_pressure:.4f}   "
      f"busy share in val: {subs['busy'].mean():.1%}")
print('subset sizes:', {k: int(v.sum()) for k, v in subs.items()})
print('\nmedians  gpu_ratio vs grp_gpu_mean:', round(ev['gpu_ratio'].median(), 3), round(ev['grp_gpu_mean'].median(), 3),
      '| mem_ratio vs grp_mem_mean:', round(ev['mem_ratio'].median(), 3), round(ev['grp_mem_mean'].median(), 3))

show = ['all', 'hist_none', 'hist_some', 'busy', 'pool_V100_busy', 'rt_ge_1d']
show_subs = {s: subs[s] for s in show}

# Run time
y = V['log_run_time'].to_numpy()
med_rt = ev.loc[tr, 'log_run_time'].median()
rt_pred = {
    'constant (=FIFO)': np.full(len(V), med_rt),
    'tsafrir_last2 (grp->usr)': V['grp_rt_last2'].fillna(V['usr_rt_last2']).fillna(med_rt).to_numpy(),
    'oracle': y,
}
rt_tab = pd.concat({k: runtime_report(y, p, V['submit'], show_subs) for k, p in rt_pred.items()},
                   names=['baseline'])
print('\nRUN TIME (validation)')
print(rt_tab[['n', 'mae', 'kendall_hour', 'sjf_regret', 'sjf_fifo', 'sjf_gap_closed']].round(3))

# Usage
use_tabs = {}
for tgt, (h1, h2) in [('gpu_ratio', ('grp_gpu_mean', 'usr_gpu_mean')),
                      ('mem_ratio', ('grp_mem_mean', 'usr_mem_mean'))]:
    yu = V[tgt].to_numpy()
    med = ev.loc[tr, tgt].median()
    preds = {'request (=1)': np.ones(len(V)),
             'train_median': np.full(len(V), med),
             'history (grp->usr)': V[h1].fillna(V[h2]).fillna(med).to_numpy()}
    use_tabs[tgt] = pd.concat({k: usage_report(yu, p, show_subs) for k, p in preds.items()},
                              names=['baseline'])
    print(f'\n{tgt.upper()} (validation, jobs with usage data)')
    print(use_tabs[tgt][['n', 'mae', 'under_rate', 'mean_shortfall']].round(3))

rt_tab.to_csv(DRIVE / 'results/tables/baselines_val_runtime.csv')
pd.concat(use_tabs, names=['target']).to_csv(DRIVE / 'results/tables/baselines_val_usage.csv')

busy threshold (train q75 of pool_pressure): 0.0665   busy share in val: 29.6%
subset sizes: {'all': 88611, 'no_burst': 88611, 'burst': 0, 'hist_none': 26005, 'hist_some': 62606, 'busy': 26254, 'calm': 62357, 'rt_ge_1h': 20333, 'rt_ge_1d': 419, 'full_coverage': 67809, 'pool_ALL': 55544, 'pool_P100': 7441, 'pool_T4': 22117, 'pool_V100': 1972, 'pool_V100M32': 1537, 'pool_V100_busy': 1751}

medians  gpu_ratio vs grp_gpu_mean: 0.05 0.047 | mem_ratio vs grp_mem_mean: 0.169 0.194


/content/tevzi/src/tevzi/evaluation/metrics.py:56: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  'spearman': float(stats.spearmanr(y, p).statistic) if len(y) > 1 else np.nan,



RUN TIME (validation)
                                             n    mae  kendall_hour  \
baseline                 subset                                       
constant (=FIFO)         all             88611  1.780           NaN   
                         hist_none       26005  1.854           NaN   
                         hist_some       62606  1.749           NaN   
                         busy            26254  1.787           NaN   
                         pool_V100_busy   1751  1.772           NaN   
                         rt_ge_1d          419  5.635           NaN   
tsafrir_last2 (grp->usr) all             88611  0.700         0.619   
                         hist_none       26005  1.161         0.403   
                         hist_some       62606  0.508         0.669   
                         busy            26254  0.810         0.547   
                         pool_V100_busy   1751  0.788         0.560   
                         rt_ge_1d          419  2.547 

In [ ]:
# Stage 3.2: categorical features — levels and unseen-level rates (train -> val)
from tevzi.features.m1 import CATEGORICAL
print('CATEGORICAL:', CATEGORICAL)

mc = m1[['job_name'] + list(CATEGORICAL)].merge(targets[['job_name', 'period', 'purged']], on='job_name')
tr_m = (mc['period'] == 'train') & ~mc['purged']
va_m = (mc['period'] == 'val') & ~mc['purged']
rows = []
for c in CATEGORICAL:
    tr_vals = mc.loc[tr_m, c]
    counts = tr_vals.value_counts(dropna=False)
    va_vals = mc.loc[va_m, c]
    unseen = ~va_vals.isin(counts.index)
    rare = va_vals.map(counts).fillna(0) < 20
    rows.append({'feature': c, 'dtype': str(mc[c].dtype), 'levels_train': len(counts),
                 'missing_train_%': tr_vals.isna().mean() * 100,
                 'val_unseen_%': unseen.mean() * 100, 'val_rare(<20)_%': rare.mean() * 100,
                 'top_level_share_%': counts.iloc[0] / counts.sum() * 100})
print(pd.DataFrame(rows).set_index('feature').round(2))

CATEGORICAL: ['main_gpu_type', 'main_task', 'user', 'workload']
                dtype  levels_train  missing_train_%  val_unseen_%  \
feature                                                              
main_gpu_type  object             5              0.0          0.00   
main_task      object            10              0.0          0.00   
user           object           942              0.0          2.86   
workload       object            10              0.0          0.00   

               val_rare(<20)_%  top_level_share_%  
feature                                            
main_gpu_type              0.0              60.92  
main_task                  0.0              56.67  
user                       4.3              18.04  
workload                   0.0              88.07  


In [ ]:
# Stage 3.2: fit preprocessing on the training period and inspect it on validation
from tevzi.models.preprocess import CategoryMap, NumericScaler
from tevzi.features.m1 import CATEGORICAL

num_cols = [c for c in m1.columns if c not in ('job_name', *CATEGORICAL)]
d = m1.merge(targets[['job_name', 'period', 'purged']], on='job_name')
trn = d[(d['period'] == 'train') & ~d['purged']]
vl = d[(d['period'] == 'val') & ~d['purged']]

cmap = CategoryMap.fit(trn, CATEGORICAL)
print('cardinalities (incl. other):', cmap.cardinalities)
print('val share mapped to other:', (cmap.transform(vl) == 0).mean().round(4).to_dict())

scaler = NumericScaler.fit(trn, num_cols)
print(f'\n{len(num_cols)} numeric columns')
print('log1p:', scaler.log_cols)
print('missing indicators:', scaler.na_cols)
Z = scaler.transform(vl)
print('\nval matrix:', Z.shape)
print(Z.describe().T[['mean', 'std', 'min', 'max']].round(2).sort_values('max').tail(8))

cardinalities (incl. other): {'main_gpu_type': 6, 'main_task': 10, 'user': 485, 'workload': 10}
val share mapped to other: {'main_gpu_type': 0.0, 'main_task': 0.0, 'user': 0.043, 'workload': 0.0}

42 numeric columns
log1p: ['main_plan_gpu', 'main_plan_cpu', 'main_plan_mem', 'main_inst_num', 'req_gpu_total', 'req_cpu_total', 'req_mem_total', 'mem_per_gpu', 'cpu_per_gpu', 'n_ps_inst', 'total_inst', 'grp_rt_n', 'grp_rt_std', 'grp_use_n', 'grp_gpu_mean', 'grp_mem_mean', 'usr_use_n', 'usr_gpu_mean', 'usr_mem_mean', 'grp_n_prev_submit', 'grp_time_since_prev', 'grp_submit_last_1h']
missing indicators: ['grp_rt_mean', 'grp_rt_std', 'grp_rt_last', 'grp_rt_last2', 'usr_rt_mean', 'usr_rt_std', 'usr_rt_last', 'usr_rt_last2', 'grp_gpu_mean', 'grp_mem_mean', 'grp_gpu_zero_share', 'usr_gpu_mean', 'usr_mem_mean', 'usr_gpu_zero_share', 'grp_time_since_prev']

val matrix: (88611, 57)
                    mean   std   min    max
grp_submit_last_1h -0.07  0.90 -0.32   6.01
grp_gpu_mean       -0.01  0.82 -0

In [12]:
# Stage 3.3 setup: feature matrices (variants 1, 1-nouser, 1b), targets, masks, val frame
import os, time
import lightgbm as lgb
from tevzi.models.gbm import build_features, grid_search, train_seeds, SEEDS
from tevzi.features.m2 import CHANNELS
from tevzi.evaluation.metrics import sjf_regret
from tevzi.evaluation.metrics import SubsetThresholds, make_subsets
from tevzi.models.preprocess import CategoryMap
from tevzi.features.m1 import CATEGORICAL
print('lightgbm', lgb.__version__, '| CPU cores:', os.cpu_count())
assert set(CHANNELS) <= set(snap.columns)

T = targets.set_index('job_name').loc[m1['job_name']].reset_index()   # same row order as m1
tr3 = ((T['period'] == 'train') & ~T['purged']).to_numpy()
va3 = ((T['period'] == 'val') & ~T['purged']).to_numpy()
print('train submit_day range:', round(T.loc[tr3, 'submit_day'].min(), 2), round(T.loc[tr3, 'submit_day'].max(), 2))
inner = (T.loc[tr3, 'submit_day'] >= 41).to_numpy()
print(f'train {tr3.sum():,} (inner hold-out {inner.sum():,})   val {va3.sum():,}')

cmap = CategoryMap.fit(m1[tr3], CATEGORICAL)
X_all = {'1': build_features(m1, cmap),
         '1-nouser': build_features(m1, cmap, drop=('user',)),
         '1b': build_features(m1, cmap, snap=snap)}
Y = {t: T[t].to_numpy() for t in ['log_run_time', 'gpu_ratio', 'mem_ratio']}
print({k: v.shape for k, v in X_all.items()})

Vg = (T.loc[va3, ['job_name', 'log_run_time', 'gpu_ratio', 'mem_ratio', 'run_time', 'in_burst', 'usage_coverage']]
      .merge(snap[['job_name', 'submit', 'pool_key', 'pool_pressure']], on='job_name', how='left')
      .merge(m1[['job_name', 'grp_rt_n']], on='job_name', how='left'))
assert (Vg['job_name'].to_numpy() == T.loc[va3, 'job_name'].to_numpy()).all()
thr3 = SubsetThresholds.fit(snap.set_index('job_name').loc[T.loc[tr3, 'job_name'], 'pool_pressure'])
subs3 = make_subsets(Vg, thr3)
print('busy threshold:', round(thr3.busy_pressure, 4), '| V100_busy jobs:', int(subs3['pool_V100_busy'].sum()))

OUT = DRIVE / 'results/models/stage3_gbm'
OUT.mkdir(parents=True, exist_ok=True)
print(f'RAM used: {psutil.Process().memory_info().rss / 1e9:.1f} GB')

lightgbm 4.6.0 | CPU cores: 2
train submit_day range: 19.0 45.0
train 281,064 (inner hold-out 44,449)   val 88,611
{'1': (714273, 46), '1-nouser': (714273, 45), '1b': (714273, 57)}
busy threshold: 0.0665 | V100_busy jobs: 1751
RAM used: 2.8 GB


**3.3 decision criteria (written before results).** Comparison rule from
docs/DEGERLENDIRME_PROTOKOLU.md §4 (5 paired seeds; 95% CI of the paired
improvement excludes 0; relative gain ≥ 1%). Selection metric per target:
run time = sjf_regret, usage = MAE.
1. **User identity:** for each target, keep `user` (variant 1) only if 1 beats
   1-nouser on `all`; otherwise drop it.
2. **H3, GBM side:** 1b vs 1 on `pool_V100_busy` and on `all`, per target.
   H3 is supported for a target if 1b beats 1 on `pool_V100_busy` and its
   relative gain there is larger than on `all`.

In [13]:
# Stage 3.3 grid search on variant 1, one seed, no refit (resume-safe)
score_fns = {
    'log_run_time': lambda p: sjf_regret(Vg['run_time'], p, Vg['submit'])['regret'],
    'gpu_ratio': lambda p: float(np.nanmean(np.abs(p - Vg['gpu_ratio'].to_numpy()))),
    'mem_ratio': lambda p: float(np.nanmean(np.abs(p - Vg['mem_ratio'].to_numpy()))),
}
best_params = {}
X1 = X_all['1']
for tgt, fn in score_fns.items():
    f = OUT / f'grid_{tgt}.csv'
    if f.exists():
        g = pd.read_csv(f)
    else:
        t0 = time.time()
        g = grid_search(X1[tr3], Y[tgt][tr3], inner, X1[va3], fn)
        g.to_csv(f, index=False)
        print(f'{tgt} grid done in {(time.time() - t0) / 60:.1f} min')
    print(f'\n{tgt}\n', g.round(4))
    best_params[tgt] = {k: int(g.loc[0, k]) for k in ('num_leaves', 'min_child_samples')}
print('\nchosen:', best_params)


log_run_time
    setting  num_leaves  min_child_samples  best_iter   score
0        1          63                100        192  1.6516
1        0          63                 20        168  1.6687
2        3         255                100         94  1.6764
3        2         255                 20         56  1.7075

gpu_ratio
    setting  num_leaves  min_child_samples  best_iter   score
0        1          63                100         70  0.0913
1        3         255                100         37  0.0914
2        2         255                 20         13  0.1224
3        0          63                 20         13  0.1289

mem_ratio
    setting  num_leaves  min_child_samples  best_iter   score
0        0          63                 20        185  0.1689
1        2         255                 20         46  0.1760
2        1          63                100        329  0.1773
3        3         255                100        249  0.1830

chosen: {'log_run_time': {'num_leaves': 63, '

In [14]:
# Stage 3.3 final training: chosen setting, 5 seeds, every variant (resume-safe)
for tgt in score_fns:
    for var, X in X_all.items():
        f = OUT / f'pred_val_{var}_{tgt}.parquet'
        if f.exists():
            continue
        t0 = time.time()
        preds, iters, _ = train_seeds(X[tr3], Y[tgt][tr3], inner, X[va3], best_params[tgt])
        (pd.DataFrame(preds.T, columns=[f'seed{s}' for s in SEEDS])
           .assign(job_name=Vg['job_name'].to_numpy())
           .to_parquet(f, index=False))
        print(f'{tgt:12s} {var:9s} trees {iters}  {(time.time() - t0) / 60:.1f} min')
print('all done')

all done


In [15]:
# Stage 3.3 analysis: baselines vs GBM variants; paired 5-seed decisions (criteria: cell 17)
from tevzi.evaluation.metrics import paired_comparison, runtime_report, usage_report
print('chosen settings:', best_params)
KEY = ['all', 'hist_none', 'hist_some', 'busy', 'pool_V100_busy']
key_subs = {s: subs3[s] for s in KEY}
seed_cols = [f'seed{s}' for s in SEEDS]
P = {(v, t): pd.read_parquet(OUT / f'pred_val_{v}_{t}.parquet') for v in X_all for t in score_fns}
for df in P.values():
    assert (df['job_name'].to_numpy() == Vg['job_name'].to_numpy()).all()

def report(t, p):
    if t == 'log_run_time':
        r = runtime_report(Vg['log_run_time'], p, Vg['submit'], key_subs)
        return r[['mae', 'kendall_hour', 'sjf_regret', 'sjf_gap_closed']].assign(select=r['sjf_regret'])
    r = usage_report(Vg[t], p, key_subs)[['mae', 'under_rate']]
    return r.assign(select=r['mae'])

# Baselines on the same validation rows (Tsafrir for run time, history mean for usage)
bl = Vg[['job_name']].merge(m1[['job_name', 'grp_rt_last2', 'usr_rt_last2', 'grp_gpu_mean', 'usr_gpu_mean',
                                'grp_mem_mean', 'usr_mem_mean']], on='job_name', how='left')
base_pred = {
    'log_run_time': bl['grp_rt_last2'].fillna(bl['usr_rt_last2']).fillna(np.median(Y['log_run_time'][tr3])),
    'gpu_ratio': bl['grp_gpu_mean'].fillna(bl['usr_gpu_mean']).fillna(np.nanmedian(Y['gpu_ratio'][tr3])),
    'mem_ratio': bl['grp_mem_mean'].fillna(bl['usr_mem_mean']).fillna(np.nanmedian(Y['mem_ratio'][tr3])),
}

M = {}
for t in score_fns:
    rows = [report(t, base_pred[t].to_numpy()).assign(variant='baseline', seed='-')]
    for v in X_all:
        for s in seed_cols:
            rows.append(report(t, P[(v, t)][s].to_numpy()).assign(variant=v, seed=s))
    M[t] = pd.concat(rows).reset_index()
    M[t].to_csv(DRIVE / f'results/tables/gbm_val_{t}.csv', index=False)
    summary = M[t].groupby(['variant', 'subset'])['select'].agg(['mean', 'std']).unstack('subset')
    metric = 'sjf_regret' if t == 'log_run_time' else 'MAE'
    print(f'\n=== {t}: selection metric = {metric} (mean, std over seeds; lower is better) ===')
    print(summary['mean'][KEY].round(4))
    if t == 'log_run_time':
        print('\nMAE / Kendall_hour / gap_closed on all (mean over seeds):')
        print(M[t].query("subset == 'all'").groupby('variant')[['mae', 'kendall_hour', 'sjf_gap_closed']].mean().round(4))
    else:
        print('\nunder_rate on all (mean over seeds):')
        print(M[t].query("subset == 'all'").groupby('variant')['under_rate'].mean().round(4))

def compare(t, a, b, subset):
    d = M[t]
    get = lambda v: d[(d['subset'] == subset) & (d['variant'] == v)].sort_values('seed')['select'].to_numpy()
    return paired_comparison(get(a), get(b), lower_is_better=True, min_rel_gain=0.01)

print('\n=== Decision 1: keep user identity? (1 vs 1-nouser on all) ===')
for t in score_fns:
    r = compare(t, '1-nouser', '1', 'all')
    print(f"{t:12s} gain {r['rel_gain']:+.2%}  CI [{r['ci_low']:+.5f}, {r['ci_high']:+.5f}]  -> keep user: {r['b_better']}")

print('\n=== Decision 2: H3, GBM side (1b vs 1) ===')
for t in score_fns:
    v100, al = compare(t, '1', '1b', 'pool_V100_busy'), compare(t, '1', '1b', 'all')
    h3 = v100['b_better'] and v100['rel_gain'] > al['rel_gain']
    print(f"{t:12s} V100_busy gain {v100['rel_gain']:+.2%} (sig {v100['b_better']})  "
          f"all gain {al['rel_gain']:+.2%} (sig {al['b_better']})  -> H3 supported: {h3}")

chosen settings: {'log_run_time': {'num_leaves': 63, 'min_child_samples': 100}, 'gpu_ratio': {'num_leaves': 63, 'min_child_samples': 100}, 'mem_ratio': {'num_leaves': 63, 'min_child_samples': 20}}

=== log_run_time: selection metric = sjf_regret (mean, std over seeds; lower is better) ===
subset       all  hist_none  hist_some    busy  pool_V100_busy
variant                                                       
1         1.5951     1.8635     1.4052  1.6354          1.5521
1-nouser  1.6273     1.9638     1.4133  1.6278          1.6168
1b        1.6113     1.8588     1.4436  1.6079          1.5256
baseline  1.7369     2.2413     1.4434  1.7408          1.7258

MAE / Kendall_hour / gap_closed on all (mean over seeds):
             mae  kendall_hour  sjf_gap_closed
variant                                       
1         0.6670        0.6362          0.7962
1-nouser  0.6818        0.6264          0.7852
1b        0.6716        0.6347          0.7907
baseline  0.6999        0.6189        

In [16]:
# Stage 3.3 step 2a: ATLAS-style metrics on validation (context only) + inspect fit_gbm for SHAP
import inspect
from scipy.stats import spearmanr
import tevzi.models.gbm as gbm_mod

y = Vg['log_run_time'].to_numpy()
def atlas_row(variant, seed, p):
    return {'variant': variant, 'seed': seed,
            'spearman': spearmanr(y, p)[0], 'rmsle': float(np.sqrt(np.mean((p - y) ** 2)))}

rows = [atlas_row(v, s, P[(v, 'log_run_time')][s].to_numpy())
        for v in ['1', '1-nouser', '1b'] for s in seed_cols]
rows.append(atlas_row('baseline', '-', base_pred['log_run_time'].to_numpy()))
atlas = pd.DataFrame(rows)
atlas.to_csv(DRIVE / 'results/tables/gbm_val_atlas_compare.csv', index=False)
print('ATLAS reference (their test split): Spearman 0.912')
print(atlas.groupby('variant')[['spearman', 'rmsle']].agg(['mean', 'std']).round(4))

print('\n--- for the SHAP cell ---')
print('fit_gbm', inspect.signature(gbm_mod.fit_gbm))
print(inspect.getsource(gbm_mod.fit_gbm))
print('train_seeds', inspect.signature(gbm_mod.train_seeds))
print("'models' in memory:", 'models' in globals())

ATLAS reference (their test split): Spearman 0.912
         spearman           rmsle        
             mean     std    mean     std
variant                                  
1          0.8617  0.0046  1.1058  0.0175
1-nouser   0.8519  0.0054  1.1374  0.0196
1b         0.8625  0.0020  1.1051  0.0073
baseline   0.8379     NaN  1.2381     NaN

--- for the SHAP cell ---
fit_gbm (X_tr: 'pd.DataFrame', y_tr, inner, params: 'dict', seed: 'int', refit: 'bool' = True)
def fit_gbm(X_tr: pd.DataFrame, y_tr, inner, params: dict, seed: int, refit: bool = True):
    """Early-stop on the inner hold-out, then refit on all training rows.

    Rows with a missing target (e.g. no usage data) are ignored. With
    ``refit=False`` the early-stopped model itself is returned (grid search).
    Returns ``(model, best_iteration)``.
    """
    import lightgbm as lgb
    y_tr = np.asarray(y_tr, dtype=float)
    inner = np.asarray(inner, dtype=bool)
    ok = np.isfinite(y_tr)
    fit_rows, stop_rows = ok & ~i

In [17]:
# Stage 3.3 step 2b: SHAP (LightGBM pred_contrib) for the selected GBMs + 1b on GPU ratio; seed-0 refit
import time
from tevzi.models.gbm import fit_gbm

tr_mask = np.asarray(tr3, dtype=bool)
tr_inner = np.asarray(inner, dtype=bool)
if len(tr_inner) != tr_mask.sum():          # inner given over all rows -> restrict to training rows
    tr_inner = tr_inner[tr_mask]
SEL = [('1', 'log_run_time'), ('1-nouser', 'gpu_ratio'), ('1-nouser', 'mem_ratio'), ('1b', 'gpu_ratio')]

rng = np.random.default_rng(0)
n_va = len(Vg)
sample = np.zeros(n_va, dtype=bool)
sample[rng.choice(n_va, 20_000, replace=False)] = True
v100 = np.asarray(subs3['pool_V100_busy'], dtype=bool)
rows_shap = sample | v100

def group_of(c):
    if c.startswith('m2_'): return 'M2'
    if c.startswith(('grp_', 'usr_')): return 'D_history'
    if c == 'user' or 'workload' in c: return 'C_identity'
    if any(k in c for k in ('hour', 'dow', 'weekday')): return 'E_time'
    if any(k in c for k in ('n_task', 'has_ps', 'has_eval', 'ps_', 'total_inst', 'main_task')): return 'B_structure'
    return 'A_demand'

groups = {}
for c in X_all['1b'].columns:
    groups.setdefault(group_of(c), []).append(c)
print('feature groups (please check):')
for g, cs in sorted(groups.items()):
    print(f'  {g} ({len(cs)}): {cs}')

shap_tabs = {}
for v, t in SEL:
    t0 = time.time()
    Xtr, Xva = X_all[v][tr_mask], X_all[v][np.asarray(va3, dtype=bool)]
    model, best = fit_gbm(Xtr, np.asarray(Y[t])[tr_mask], tr_inner, best_params[t], seed=0)
    diff = np.max(np.abs(model.predict(Xva) - P[(v, t)]['seed0'].to_numpy()))
    a = np.abs(model.predict(Xva[rows_shap], pred_contrib=True)[:, :-1])
    in_samp, in_v100 = sample[rows_shap], v100[rows_shap]
    tab = pd.DataFrame({'feature': Xva.columns, 'group': [group_of(c) for c in Xva.columns],
                        'mean_abs_all': a[in_samp].mean(0), 'mean_abs_v100busy': a[in_v100].mean(0)})
    tab = tab.sort_values('mean_abs_all', ascending=False)
    tab.to_csv(DRIVE / f'results/tables/gbm_shap_{v}_{t}.csv', index=False)
    shap_tabs[(v, t)] = tab
    g = tab.groupby('group')[['mean_abs_all', 'mean_abs_v100busy']].sum()
    print(f'\n=== {v} / {t}: best_iter {best}, max |pred - saved seed0| = {diff:.1e}, {time.time() - t0:.0f} s ===')
    print(tab.head(12).round(4).to_string(index=False))
    print('group share of mean |SHAP|:')
    print((g / g.sum()).round(3).to_string())

feature groups (please check):
  A_demand (11): ['main_plan_gpu', 'main_plan_cpu', 'main_plan_mem', 'main_inst_num', 'req_gpu_total', 'req_cpu_total', 'req_mem_total', 'gpu_fractional', 'mem_per_gpu', 'cpu_per_gpu', 'main_gpu_type']
  B_structure (6): ['n_tasks', 'has_ps', 'has_evaluator', 'n_ps_inst', 'total_inst', 'main_task']
  C_identity (3): ['workload_missing', 'user', 'workload']
  D_history (22): ['grp_rt_n', 'grp_rt_mean', 'grp_rt_std', 'grp_rt_last', 'grp_rt_last2', 'usr_rt_n', 'usr_rt_mean', 'usr_rt_std', 'usr_rt_last', 'usr_rt_last2', 'grp_use_n', 'grp_gpu_mean', 'grp_mem_mean', 'grp_gpu_zero_share', 'usr_use_n', 'usr_gpu_mean', 'usr_mem_mean', 'usr_gpu_zero_share', 'grp_n_prev_submit', 'grp_seen_before', 'grp_time_since_prev', 'grp_submit_last_1h']
  E_time (4): ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
  M2 (11): ['m2_arr_count', 'm2_arr_gpu', 'm2_occ_gpu', 'm2_occ_cpu', 'm2_occ_mem', 'm2_pressure', 'm2_use_gpu', 'm2_pool_occ', 'm2_pool_pressure', 'm2_pool_arr_gpu', 

In [18]:
# Stage 3.3 step 3 (decision 14 cost): run-time GBM trained on days 7-45 vs 19-45 (variant 1, 5 seeds)
from tevzi.evaluation.metrics import paired_comparison
print(T['period'].value_counts().to_string())

end_day = T['submit_day'] + (T['wait_time'] + T['run_time']) / 86400
assert np.isfinite(end_day.to_numpy()).all()
in_early = ((T['submit_day'] >= 7) & (T['submit_day'] < 19)).to_numpy()
early = in_early & (end_day < 45).to_numpy()            # same purge rule as the train period
assert not (early & (tr3 | va3)).any()
tr7 = tr3 | early
print(f'early rows added {early.sum():,} (purged: {(in_early & ~early).sum():,})  |  train 19-45 {tr3.sum():,} -> 7-45 {tr7.sum():,}')

cmap7 = CategoryMap.fit(m1[tr7], CATEGORICAL)            # fit only on the (longer) training window
X7 = build_features(m1, cmap7)
inner7 = (T.loc[tr7, 'submit_day'] >= 41).to_numpy()
assert inner7.sum() == inner.sum()

t0 = time.time()
pred7, iters7, _ = train_seeds(X7[tr7], Y['log_run_time'][tr7], inner7, X7[va3],
                               best_params['log_run_time'], seeds=SEEDS)
print(f'trees 7-45: {list(iters7)}   ({time.time() - t0:.0f} s)')
(pd.DataFrame(np.asarray(pred7).T, columns=seed_cols).assign(job_name=Vg['job_name'].to_numpy())
   .to_parquet(OUT / 'pred_val_1-day7_log_run_time.parquet', index=False))

rows = []
runs = {'19-45': [P[('1', 'log_run_time')][s].to_numpy() for s in seed_cols], '7-45': list(pred7)}
for name, preds in runs.items():
    for s, p in zip(seed_cols, preds):
        rows.append(report('log_run_time', p).assign(train=name, seed=s))
D14 = pd.concat(rows).reset_index()
D14.to_csv(DRIVE / 'results/tables/gbm_val_decision14.csv', index=False)
print(D14[D14['subset'].isin(['all', 'hist_none', 'hist_some'])]
      .groupby(['subset', 'train'])[['select', 'mae', 'kendall_hour']].mean().round(4))

res = {}
for sub in ['all', 'hist_none']:
    d = D14[D14['subset'] == sub]
    get = lambda n: d[d['train'] == n].sort_values('seed')['select'].to_numpy()
    res[sub] = r = paired_comparison(get('19-45'), get('7-45'), lower_is_better=True, min_rel_gain=0.01)
    print(f"{sub:10s} 7-45 vs 19-45: gain {r['rel_gain']:+.2%}  CI [{r['ci_low']:+.5f}, {r['ci_high']:+.5f}]  sig {r['b_better']}")
print('\nPre-registered: decision stays 19-45. Open scope question?',
      bool(res['all']['b_better'] and res['all']['rel_gain'] >= 0.03))

period
train     281595
test      219435
warmup    123831
val        89412
early rows added 122,623 (purged: 0)  |  train 19-45 281,064 -> 7-45 403,687
trees 7-45: [179, 137, 151, 171, 310]   (224 s)
                 select     mae  kendall_hour
subset    train                              
all       19-45  1.5951  0.6670        0.6362
          7-45   1.5498  0.6627        0.6347
hist_none 19-45  1.8635  0.9982        0.4594
          7-45   1.8306  0.9875        0.4616
hist_some 19-45  1.4052  0.5294        0.6707
          7-45   1.3661  0.5278        0.6665
all        7-45 vs 19-45: gain +2.85%  CI [+0.03120, +0.05957]  sig True
hist_none  7-45 vs 19-45: gain +1.77%  CI [-0.00621, +0.07202]  sig False

Pre-registered: decision stays 19-45. Open scope question? False
